# B3 — Chargement de conteneurs : Bin Packing 3D
**Emile Jouannet — EPITA SCIA 2026**

---

## Problème

Le **3D Bin Packing Problem** consiste à placer un ensemble d'objets parallélépipédiques dans un **nombre minimal de conteneurs**, en respectant :
- Non-chevauchement géométrique des objets
- Limites dimensionnelles du conteneur
- Contraintes optionnelles : fragilité, poids maximum

**Complexité** : NP-difficile (réduction depuis Partition). Pas d'algorithme polynomial connu.

**Applications** : logistique (Amazon, DHL), cloud (VMs), jeux vidéo (packing d'assets).

In [ ]:
import sys
sys.path.insert(0, '..')

from src.model import Item, Container, solve
from src.heuristic import first_fit_decreasing
from src.visualization import plot_bin, plot_all_bins, summary_table
from src.benchmarks import REFERENCE_INSTANCES, generate_instance, scalability_suite

import pandas as pd
import plotly.graph_objects as go

---
## 1. Modélisation CP-SAT

### Variables
| Variable | Type | Domaine | Signification |
|----------|------|---------|---------------|
| `bin_var[i]` | IntVar | 0..n-1 | Quel conteneur reçoit l'objet i |
| `x[i], y[i], z[i]` | IntVar | 0..W-w, ... | Position dans le conteneur |

### Contrainte de non-chevauchement

Pour chaque paire (i, j) dans le même conteneur, au moins une des 6 séparations axiales doit tenir :

$$x_i + w_i \leq x_j \;\text{OU}\; x_j + w_j \leq x_i \;\text{OU}\; y_i + d_i \leq y_j \;\text{OU}\; \ldots$$

Encodé avec 6 booléens auxiliaires `sep[k]` et `AddBoolOr`.

### Cassage de symétrie
Les indices de conteneurs sont ordonnés : `bin[0]=0`, `bin[i] ≤ bin[i-1]+1`.  
Sans ça, permuter deux conteneurs vides = même solution → **explosion combinatoire**.

### Objectif : $\min\, (\max_i\, \text{bin\_var}[i] + 1)$

---
## 2. Exemple simple : visualisation 3D

In [ ]:
inst = REFERENCE_INSTANCES['small_5']
container, items = inst['container'], inst['items']

print(f'Conteneur : {container.W}x{container.D}x{container.H}')
for i, it in enumerate(items):
    print(f'  Objet {i} : {it.w}x{it.d}x{it.h}  vol={it.volume}')

In [ ]:
result = solve(items, container, time_limit=30)
print(f'Statut       : {result["status"]}')
print(f'Bins utilisés : {result["num_bins"]}')
print(f'Temps        : {result["solve_time"]:.3f}s')
print(f'Assignation  : {result["assignment"]}')
print(f'Positions    : {result["positions"]}')

In [ ]:
fig = plot_bin(items, result, bin_idx=0, container=container)
fig.show()

---
## 3. CP-SAT vs Heuristique (First Fit Decreasing)

La **FFD** trie par volume décroissant et place dans le premier bin disponible. O(n log n) mais sous-optimale. CP-SAT garantit l'optimalité.

In [ ]:
rows = []
for name, inst in REFERENCE_INSTANCES.items():
    c, it = inst['container'], inst['items']
    r_cp   = solve(it, c, time_limit=30)
    r_heur = first_fit_decreasing(it, c)
    rows.append({
        'Instance'        : name,
        'n'               : len(it),
        'CP-SAT bins'     : r_cp['num_bins'],
        'FFD bins'        : r_heur['num_bins'],
        'Optimal connu'   : inst.get('opt', '?'),
        'Temps (s)'       : round(r_cp['solve_time'], 3),
        'Statut'          : r_cp['status'],
    })

df = pd.DataFrame(rows)
df

---
## 4. Scalabilité

In [ ]:
container = Container(W=20, D=20, H=20)
suite = scalability_suite(container, sizes=[5, 8, 10, 12, 15])

scale_rows = []
for n, items in suite:
    r  = solve(items, container, time_limit=60)
    rh = first_fit_decreasing(items, container)
    scale_rows.append({'n': n, 'CP-SAT bins': r['num_bins'],
                        'FFD bins': rh['num_bins'],
                        'temps (s)': round(r['solve_time'],3), 'statut': r['status']})
    print(f'n={n:2d} -> {r["num_bins"]} bins  ({r["solve_time"]:.3f}s)  [{r["status"]}]')

df_scale = pd.DataFrame(scale_rows)
df_scale

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_scale['n'], y=df_scale['temps (s)'],
    mode='lines+markers', name='Temps CP-SAT', line=dict(color='royalblue', width=2)))
fig.update_layout(title='Temps de résolution CP-SAT vs n',
    xaxis_title="Nombre d'objets", yaxis_title='Temps (s)', template='plotly_white')
fig.show()

In [ ]:
fig2 = go.Figure([
    go.Bar(x=df_scale['n'], y=df_scale['CP-SAT bins'], name='CP-SAT', marker_color='royalblue'),
    go.Bar(x=df_scale['n'], y=df_scale['FFD bins'],    name='FFD',    marker_color='tomato'),
])
fig2.update_layout(barmode='group', title='CP-SAT vs FFD — bins utilisés',
    xaxis_title="n objets", yaxis_title='Bins', template='plotly_white')
fig2.show()

---
## 5. Contraintes avancées
### 5a. Fragilité

In [ ]:
inst_f = REFERENCE_INSTANCES['fragile_4']
r_f = solve(inst_f['items'], inst_f['container'], time_limit=30)
print(f'{r_f["num_bins"]} bin(s) | {r_f["status"]}')
for i, it in enumerate(inst_f['items']):
    pz = r_f['positions'][i][2]
    top = (pz == inst_f['container'].H - it.h)
    print(f'  Objet {i}{" [FRAGILE]" if it.fragile else ""} : z={pz}, au sommet={top}')
plot_bin(inst_f['items'], r_f, 0, inst_f['container']).show()

### 5b. Poids maximum par conteneur

In [ ]:
inst_w = REFERENCE_INSTANCES['weight_6']
r_w = solve(inst_w['items'], inst_w['container'], time_limit=30)
print(f'{r_w["num_bins"]} bin(s) | poids max = {inst_w["container"].max_weight}')
for b in range(r_w['num_bins']):
    poids = sum(inst_w['items'][i].weight for i,a in enumerate(r_w['assignment']) if a==b)
    print(f'  Bin {b} : {poids:.1f} kg <= {inst_w["container"].max_weight}')

---
## 6. Visualisation solution complète (medium_10)

In [ ]:
inst_m = REFERENCE_INSTANCES['medium_10']
r_m = solve(inst_m['items'], inst_m['container'], time_limit=60)
print(summary_table(inst_m['items'], r_m, inst_m['container']))
for fig in plot_all_bins(inst_m['items'], r_m, inst_m['container']):
    fig.show()

---
## 7. Limites et perspectives

| Limite | Explication |
|--------|-------------|
| **Scalabilité** | O(n²) paires → lent au-delà de ~20 objets |
| **Orientations** | 6 rotations possibles non modélisées |
| **Stabilité** | Centre de gravité non contraint |

### Pistes
1. **Orientations** : 6 booléens de rotation par objet
2. **Décomposition** : recherche dichotomique sur le nombre de bins
3. **Warm start** : initialiser CP-SAT avec la solution FFD
4. **Grandes instances** : ALNS ou génération de colonnes (n > 50)